# Entraîner le classificateur sur Vast.ai

Ce notebook entraîne le classificateur InceptionV3 du projet en deux phases : extraction des caractéristiques puis fine-tuning. Il réutilise le prétraitement de `src/` et utilise `sparse_categorical_crossentropy`.

## Préparer l'instance Vast.ai
1. Crée une instance avec GPU, stockage persistant (volume) et une image Jupyter/Python compatible avec TensorFlow et CUDA. Vérifie que le GPU est visible avant de lancer l'entraînement.
2. Place ou clone **tout le projet** dans le volume persistant. Envoie aussi le dataset dans `data/raw` (ou définis `DATA_DIR` dans la cellule de configuration).
3. Les images doivent être rangées par classe, par exemple `data/raw/Train/Glioma/images/...` et `data/raw/Val/Glioma/images/...`. Les fichiers source restent inchangés.
4. Ouvre ce notebook depuis le dossier du projet et exécute les cellules dans l'ordre. Si ton projet est ailleurs, définis `PROJECT_ROOT` dans l'environnement avant de démarrer Jupyter.

Les résultats sont écrits sous `models/` dans le projet (ou dans le chemin défini par `MODELS_DIR`). Garde le projet et les résultats sur le volume persistant, puis télécharge `final_model.keras`, `class_names.json` et les métriques avant de détruire l'instance. Le notebook ne reprend pas une époque interrompue; il conserve les checkpoints de phase pour pouvoir reprendre manuellement.

In [ ]:
import json
import os
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

configured_root = os.environ.get("PROJECT_ROOT")
root_candidates = (
    [Path(configured_root).expanduser()]
    if configured_root
    else [Path.cwd(), *Path.cwd().parents]
)
PROJECT_ROOT = next(
    (path.resolve() for path in root_candidates if (path / "src").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Racine du projet introuvable. Ouvre le notebook depuis le projet ou définis PROJECT_ROOT."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = Path(os.environ.get("DATA_DIR", PROJECT_ROOT / "data" / "raw")).expanduser().resolve()
MODELS_DIR = Path(os.environ.get("MODELS_DIR", PROJECT_ROOT / "models")).expanduser().resolve()
MODELS_DIR.mkdir(parents=True, exist_ok=True)

from src.data_preprocessing import (
    creer_datasets_images,
    inspecter_dataset_images,
    repartir_images,
)
from src.model_architecture import build_inception_model, prepare_for_fine_tuning

SEED = 42
IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS_FEATURE_EXTRACTION = 10
EPOCHS_FINE_TUNING = 10
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow :", tf.__version__)
print("GPU(s) :", gpus)
print("Projet :", PROJECT_ROOT)
print("Dataset :", DATA_DIR)
print("Sortie :", MODELS_DIR)
if not gpus:
    raise RuntimeError(
        "Aucun GPU TensorFlow n'est visible. Vérifie l'image CUDA de l'instance Vast.ai et son accès GPU."
    )
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Dossier de données introuvable : {DATA_DIR}")

In [ ]:
images, invalid_files = inspecter_dataset_images(DATA_DIR)
if images.empty:
    raise ValueError(
        f"Aucune image valide trouvée dans {DATA_DIR}. Vérifie que les images sont rangées par dossier de classe."
    )
print(f"Images valides : {len(images)}")
print(f"Fichiers invalides/ignorés : {len(invalid_files)}")
display(images.groupby("label").size().rename("images").to_frame())
if not invalid_files.empty:
    display(invalid_files.head(10))

splits = repartir_images(
    images, taille_test=0.15, taille_validation=0.15, seed=SEED
)
datasets, split_workspace, class_names, split_counts = creer_datasets_images(
    splits,
    taille_image=IMAGE_SIZE,
    taille_lot=BATCH_SIZE,
    seed=SEED,
    augmenter=True,
    equilibrer=True,
)
print("Classes et indices :", dict(enumerate(class_names)))
print("Images par split avant équilibrage :", {name: len(split) for name, split in splits.items()})
print("Effectifs de chargement :", split_counts)
print("Split stratifié 70/15/15; augmentation et équilibrage appliqués uniquement à Train.")

class_names_path = MODELS_DIR / "class_names.json"
class_names_path.write_text(
    json.dumps(class_names, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Correspondance des classes enregistrée :", class_names_path)

## Phase 1 — Extraction des caractéristiques

La base InceptionV3 pré-entraînée reste gelée; seule la tête de classification est entraînée.

In [ ]:
model, base_model = build_inception_model(
    num_classes=len(class_names), input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3)
)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

phase1_path = MODELS_DIR / "phase1_best.keras"
phase1_callbacks = [
    ModelCheckpoint(
        phase1_path,
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
    EarlyStopping(
        monitor="val_loss", patience=4, restore_best_weights=True, verbose=1
    ),
]
history_phase1 = model.fit(
    datasets["train"],
    validation_data=datasets["validation"],
    epochs=EPOCHS_FEATURE_EXTRACTION,
    callbacks=phase1_callbacks,
)
phase1_best_val_accuracy = max(history_phase1.history["val_accuracy"])
print(f"Meilleure accuracy validation — phase 1 : {phase1_best_val_accuracy:.4f}")

## Phase 2 — Fine-tuning

Les dernières couches convolutionnelles sont dégelées avec un taux d'apprentissage réduit. Les couches BatchNormalization restent gelées.

In [ ]:
model = prepare_for_fine_tuning(model, base_model, unfreeze_from_layer=249)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

phase2_path = MODELS_DIR / "phase2_best.keras"
phase2_callbacks = [
    ModelCheckpoint(
        phase2_path,
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
    EarlyStopping(
        monitor="val_loss", patience=4, restore_best_weights=True, verbose=1
    ),
]
history_phase2 = model.fit(
    datasets["train"],
    validation_data=datasets["validation"],
    epochs=EPOCHS_FINE_TUNING,
    callbacks=phase2_callbacks,
)
phase2_best_val_accuracy = max(history_phase2.history["val_accuracy"])

best_checkpoint = (
    phase2_path
    if phase2_best_val_accuracy >= phase1_best_val_accuracy
    else phase1_path
)
model = tf.keras.models.load_model(best_checkpoint)
final_model_path = MODELS_DIR / "final_model.keras"
model.save(final_model_path)
print(f"Meilleure accuracy validation — phase 2 : {phase2_best_val_accuracy:.4f}")
print("Checkpoint sélectionné :", best_checkpoint)
print("Modèle final enregistré :", final_model_path)

## Évaluation finale sur Test

Test n'est utilisé qu'après le choix du checkpoint. Le rapport, la matrice de confusion et les métriques sont enregistrés dans le dossier de sortie.

In [ ]:
test_metrics = model.evaluate(datasets["test"], verbose=1, return_dict=True)
y_true = np.concatenate([labels.numpy() for _, labels in datasets["test"]])
y_probabilities = model.predict(datasets["test"], verbose=1)
y_pred = np.argmax(y_probabilities, axis=1)

report = classification_report(
    y_true,
    y_pred,
    labels=np.arange(len(class_names)),
    target_names=class_names,
    output_dict=True,
    zero_division=0,
)
display(pd.DataFrame(report).transpose().round(4))

confusion = confusion_matrix(y_true, y_pred, labels=np.arange(len(class_names)))
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    confusion,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    ax=ax,
)
ax.set_title("Matrice de confusion — Test")
ax.set_xlabel("Prédit")
ax.set_ylabel("Réel")
fig.tight_layout()
confusion_path = MODELS_DIR / "confusion_matrix.png"
fig.savefig(confusion_path, dpi=150)
plt.show()
plt.close(fig)

metrics_path = MODELS_DIR / "test_metrics.json"
metrics_path.write_text(
    json.dumps({key: float(value) for key, value in test_metrics.items()}, indent=2),
    encoding="utf-8",
)
report_path = MODELS_DIR / "classification_report.csv"
pd.DataFrame(report).transpose().to_csv(report_path, encoding="utf-8")
split_workspace.cleanup()

print("Métriques Test :", {key: round(float(value), 4) for key, value in test_metrics.items()})
print("Artefacts :", final_model_path, class_names_path, metrics_path, report_path, confusion_path, sep="\n- ")

## Récupérer les résultats

Dans Jupyter sur Vast.ai, navigue jusqu'au dossier `models/` et télécharge les fichiers voulus. Pour les conserver après l'arrêt de l'instance, vérifie d'abord que le dossier se trouve sur le volume persistant. Les fichiers utiles à l'application sont `final_model.keras` et `class_names.json`; garde aussi `test_metrics.json`, `classification_report.csv` et `confusion_matrix.png` pour les résultats.